# 정당성 검정 - 각 사건 별 유의미성

SM

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from datetime import timedelta

# -----------------------------
# 사용자 설정
event_window_size = 7  # 사건일 전후 영업일 수
normal_start_date = pd.to_datetime('2022-01-01')  # 코로나 이후 정상 시기 시작일
# -----------------------------

# 결과 저장 리스트
results = []

# 사건 데이터 로드
incidents_df = pd.read_csv("../데이터/병합_사건_전처리.csv", parse_dates=['사건 날짜'])

# KOSDAQ150 벤치마크 데이터 로드
benchmark_df = pd.read_csv("KOSDAQ150.csv", parse_dates=['date'])
benchmark_df = benchmark_df.sort_values('date').set_index('date')
benchmark_df['return'] = np.log(benchmark_df['close'] / benchmark_df['close'].shift(1))

# 사건 순회
for idx, row in incidents_df.iterrows():
    agency = row['소속사']
    incident_date = row['사건 날짜']
    incident_name = row['사건 내용']

    # 소속사 주가 데이터 로드
    try:
        stock_df = pd.read_csv(f"../데이터/{agency}_주가_전처리.csv", parse_dates=['날짜'])
    except FileNotFoundError:
        continue  # 파일 없으면 건너뛰기

    stock_df = stock_df.sort_values('날짜').set_index('날짜')
    stock_df['return'] = np.log(stock_df['종가'] / stock_df['종가'].shift(1))

    # incident_date가 인덱스에 없으면 skip
    if incident_date not in stock_df.index or incident_date not in benchmark_df.index:
        continue

    # incident_date의 위치 인덱스
    try:
        center_idx = stock_df.index.get_loc(incident_date)
    except KeyError:
        continue

    # 이벤트 윈도우 범위 지정 (±7행)
    start_idx = center_idx - event_window_size
    end_idx = center_idx + event_window_size + 1

    if start_idx < 0 or end_idx > len(stock_df):
        continue  # 윈도우 벗어나면 skip

    event_returns = stock_df.iloc[start_idx:end_idx]['return']
    event_benchmark = benchmark_df.loc[event_returns.index]['return']
    event_ar = event_returns.values - event_benchmark.values
    event_car = np.sum(event_ar)

    # 정상 시기 AR 분포 추출
    normal_returns = stock_df.loc[normal_start_date:incident_date - timedelta(days=1)]['return']
    normal_benchmark = benchmark_df.loc[normal_returns.index]['return']
    normal_ar = normal_returns.values - normal_benchmark.values

    # 검정 수행
    if len(normal_ar) < 10:
        continue  # 너무 적으면 skip

    t_stat, p_value = stats.ttest_1samp(normal_ar, event_car)

    results.append({
        '소속사': agency,
        '사건명': incident_name,
        '사건일': incident_date,
        'CAR': event_car,
        't통계량': t_stat,
        'p값': p_value
    })

# 결과 데이터프레임화
results_df = pd.DataFrame(results)
results_df


FileNotFoundError: [Errno 2] No such file or directory: 'KOSDAQ150.csv'

YG

JYP

HYBE

RBW

SM C&C

큐브

키이스트